In [44]:
#Generator maili outreachowych
#Co robi:
#Dajesz nazwę firmy, branżę i krótki opis czym się zajmują. Bot generuje spersonalizowanego maila partnerskiego do tej firmy w imieniu Last Agency.
#  Evaluator ocenia czy mail jest konkretny i nie brzmi jak spam. Jeśli nie przejdzie — rerun z feedbackiem.

#Tools:
#save_email(company, email_content, subject) — zapisuje zatwierdzony mail do CSV
#flag_generic_email(company, reason) — zapisuje odrzucony mail z powodem żebyś wiedział co poprawić

#Evaluator ocenia:

#Czy mail odnosi się do konkretnej branży/usług firmy
#Czy ma jasny hook — dlaczego akurat ta firma miałaby być partnerem
#Czy nie brzmi jak masowy szablon
#Czy jest po polsku i ma naturalny ton

#Importy + OpenAI — masz już
#save_email — zapis do CSV, masz już wzorzec
#flag_generic_email — zapis do CSV
#JSONy dla obu tools
#handle_tool_calls — identycznie jak w poprzednim projekcie
#system_prompt dla generatora
#evaluator_system_prompt
#klasa Evaluation
#evaluate
#rerun
#chat

In [45]:
#importy 

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
import os
import json
import csv


In [46]:

openai = OpenAI()

In [47]:
# piszemy 1 funkcje tools która zapisuje mail do csv

def save_email(company,email_content,subject):

    # chcemy zapisac do csv mail z info cpmpamy, email content i temat
    # otwieramy csv mail metodą with open wskazujemy plik, dodajemy a kótre oznacza append = apdisz newline, przejscie do nowej lini to wwyszko zapisujemy jako obiekt f

    with open('mail.csv' , 'a' , newline=''  , encoding='utf-8') as f: 

        # tworzymy obiekt mail który zawiera klase csv z impirtu z metoa writer kóra tworzy obiekt mogoący nadpisywac pliki csf, jako paramtr przekazujemy utworzony oniekt f
        mail = csv.writer(f) 

        # teraz na obiekcie mail uzywamy metody writerow, która nadpisuje plikc csv o przekazane argumenty // argumenty zamieniamy na arr// liste

        mail.writerow([company,email_content,subject])

        # zwracamy info do modelu ze narzedzie zostało uzyte

    return{'recorded': 'ok' }



In [48]:
# 2 funkcja tools która zapisze odrzuconmy mail z jasnym powodem zeby było wiadomo co poprawić i wrzuca do csv info

def repeat_mail(company,reason):

    # otwieramy csv metodą with open i zapisujemy plik ajko obiekt f

    with open('repeatmail.csv' , 'a' , newline='', encoding='utf-8' ) as f:

        # za pomocą blbioteki csv metoa writer która pozwala napdisywać plik (obiekt f) tworzymy obiekt mail

        mail = csv.writer(f)

        # teraz obiekt mail odpalamy z metodą writerow, która bedzie anpdisywac plik i rpzekazujemy argumenty z funkcji

        mail.writerow([company,reason])

    # zwracamy info do moelu ze tools został uzyty

    return{'recorded' : 'ok'}

    



In [49]:
# teraz piszemy json tak jak wymaga tego api, to dzieki niemu tools dziala 
# kazdy wymaga name description parametrs  ktory ma type object i properties które zaczyna sie od nazwy paramtetru który ma type i description, required które jest listą properties i additional

save_email_json = {

    'name' : "save_email",
    'description' : 'zapsisuje mail do pliku csv ',
    # dodajemy parametrs czyli parametry funkcji narzędzia
    "parameters" : {
        'type' : 'object',
        'properties' : {
            'company' : {
                'type' : 'string',
                "description" : 'zapisuje nazwę firmy'
            },
            'email_content' : {
                'type' : 'string',
                'description' : 'zapisuje treść maila'
            },
            'subject' : {
                'type' : 'string',
                'description': 'zapisuje temat maila'
            }
        },
        'required' : ['company',"email_content",'subject'],
        "additionalProperties": False

    }
       

}

In [50]:
# 2 tools

repeat_mail_json = {
    'name' : 'repeat_mail',
    'description' : 'zapisuje mail do pliku csv z nazwą firmy i powodem odrzutu maila',
    'parameters' : {

        'type' : 'object',
        'properties' : {
            'company' : {

                'type' : 'string',
                'description': 'zapisuje nazwe firmy'

            },
            'reason' : {
                'type' : 'string',
                'description' : 'podaje powód przez który treść maila zostałą odrzucona'
            },

        },
         'required' : ['company','reason'],
         "additionalProperties": False


    }
}

In [51]:
# tworzymy listę tools, gdzie dajemy type function i function nazwe json
tools = [

    {'type' : 'function' , 'function' : save_email_json},
    {'type' : 'function' , 'function' : repeat_mail_json}
]


In [52]:

# definiujemy funkcje handle tools która pozwala modelowi ai pobierac info o tools i je uruchamiac
# tool_calls to parametr którego sami nie tworzymy to To jest parametr tworzony przez openAI api 

def handle_tool_calls(tool_calls):

    # to będzie nasza tablica do ktorej wrzucamy elemęty z zapetlonej tablicy narzędzi, jesli model bedzie chciał uzyc któregos z narzędzi

    results = []

    # teraz bedziemy zapetlac tablice tool_calls z narzedziami, kod wykonujemy na pojedynczym narzedziu z llisty

    for tool_call in tool_calls:

        # pobieramy parametr function.name niezbedny do uruchomienia funkcji
        tool_name = tool_call.function.name
        # pobieramy argumenty z funkcji ale z jasona   czyli json.loads jest po to zeby do parametrów dodac argumenty w formie obiekty key- value np jesli mamy company to json loads zaminei je na 
        #save_email(company="Tebim", email_content="Cześć...", subject="Współpraca")



        arguments = json.loads(tool_call.function.arguments) 


        # teraz robimy ify jesli model uruchomi dane narzędzie ( tzn jesli jego tool)_name jest zgodny z nazwą funkcji to funkcje (narzedzie przekazemy do zmiennej result)

        if tool_name == 'save_email':
            ## wywolujemy daną funkcje i przekazujemy ją do zmiennej wraz z listą argumentów key- value dzieki json.loads

            result = save_email(**arguments)
        
        elif tool_name == 'repeat_mail':

            result = repeat_mail(**arguments)

        # teraz do tablicy results która jest zwracana z funnkcji

        # przekazujemy info zogdnie z tym czego rzada api czyli role " tool" content  wynik funkcji która jest przekazana do reslut             result = save_email(**arguments) zamieniamy na json
        # i przekazujemy id danego nardzedzia z listy tool
        results.append(
            {

                'role' : 'tool',
                'content' : json.dumps(result),
                'tool_call_id' : tool_call.id

            }
        )

    # zwracamy z funkci tablice results 
    return results

In [53]:
system_prompt = """Jesteś asystentem do generowania maili outreachowych dla Last Agency - polskiej agencji SEO/SEM/GEO/AI Search z Poznania.

Dostajesz nazwę firmy, branżę i krótki opis czym się zajmują. Twoim zadaniem jest napisanie spersonalizowanego maila partnerskiego do tej firmy.

Jak działasz:
1. Na podstawie podanych informacji napisz spersonalizowanego maila partnerskiego
2. Mail musi być po polsku, naturalny i konkretny
3. Jeśli mail jest dobry - użyj narzędzia save_email żeby go zapisać
4. Jeśli mail jest zbyt ogólny lub szablonowy - użyj repeat_mail z powodem

Dobry mail:
- Odnosi się do konkretnej branży i usług firmy
- Ma jasny hook - dlaczego akurat ta firma miałaby być partnerem Last Agency
- Nie brzmi jak masowy szablon
- Jest krótki, konkretny i po polsku
- Proponuje konkretny model współpracy - white-label lub referral"""


In [54]:
evaluator_system_prompt = """Oceniasz czy wygenerowany mail outreachowy jest dobry i nadaje się do wysłania przez Last Agency - polską agencję SEO/SEM/GEO/AI Search z Poznania.

Dobry mail:
- Jest po polsku i ma naturalny ton
- Odnosi się do konkretnej branży i usług firmy do której jest wysyłany
- Ma jasny hook - konkretny powód dlaczego ta firma miałaby być partnerem Last Agency
- Nie brzmi jak masowy szablon
- Proponuje konkretny model współpracy - white-label lub referral
- Jest krótki i konkretny

Odrzuć mail jeśli:
- Brzmi jak szablon który można wysłać do każdej firmy
- Nie odnosi się do konkretnych usług lub branży firmy
- Jest po angielsku
- Nie ma konkretnego hooka

Zwróć TYLKO JSON z polami:
- is_acceptable (bool): czy mail nadaje się do wysłania
- feedback (string): co poprawić jeśli mail jest odrzucony"""


In [55]:


# tworzymy klase BaseModel zaiportowaną z pydantica / model evalautor który otrzymuje do analizy zgodnie z evalautor system prompt odpowiedz na zapytanie z 1 modelu chat
# analizuje odpowiedz i daje swoja a ta odpowiedz z modelu evalautor jest przesylanad o obiektu basemodel i na jej podstawie tworozny jest obiekt który zwraca true / false w 
# zaleznosci czy model evalautor uzna odpoiwedz za dobrą czy tez sła i feedback, czyli daje informacje co model rerun ma poprawić 


class Evaluation(BaseModel):
    is_acceptable: bool
    feedback : str


In [56]:
# tworzymy funkcje evalautor która przyjmuje reply, czyli zmienną z odpowiedzia modelu z funkcji chat, message czyli liste wiadomosci i history czyli obiekt z histora 
# pytani usera i odpowiedzi modelu i tez zwracamy odpowiedz moedlu do evaluation // funkcja odpala się w głownej funkcji chat

def evaluate(reply,message,history) -> Evaluation:

    # wpierw towrzymy liste messages która przekazemy do modelu openai w tej funkcji
    messages = [
    {'role': 'system', 'content': evaluator_system_prompt},
    {'role': 'user', 'content': f"Firma: {message}\n\nWygenerowany mail:\n{reply}"}

    ]

    # teraz budujemy model który zwróci odpowiedz ale musmimy uzyc metody parse, która stworzy nam obiekt, z danych a nie string, zeby mozna było odpowiedzi uzyc w base model i tez musimy dodac key beta
    # dodajemy tez response format i przypisujemy do neigo clase Evaluation

    response = openai.beta.chat.completions.parse(

        model = 'gpt-4.1-mini',
        messages=messages,
        response_format = Evaluation

    )

    # zwracamy odpowiedz do modelu evaluation z metoda parsed, która zwraca wymagany obiekt 

    return response.choices[0].message.parsed

In [57]:
# teraz piszemy model rerun który odpala się w głownej funkcju chat jesli model evalautor uzna odpowiedz 1 modelu za błedną 
# funkcja wówczas jeszcze raz odpala tools i odpowida na zadane pytanie 
# funkcja przyjmuje jako parametr reply, message, history, feedback z modelu evaluate z pydantica 


def rerun (reply,message,history,feedback): 

    # piszemy nowy prompt, który zawiera stary prompt,, reply , message,

    new_system_prompt = system_prompt + f'poprzednia wiadomosc zostala odrzucona'
    new_system_prompt += f'tutaj masz wiadomość która została odrzucona {reply}'
    new_system_prompt += f'tutaj masz pytanie usera {message}'
    new_system_prompt += f'tutaj masz feedback od evaluaotra dlaczego została odrzucona ta odpowiedź {feedback}' 

    # budujemy messages dla moedlu

    messages = [{'role' :'system', "content" : new_system_prompt }] + history + [{'role' : 'user' , 'content' : message}]

    # teraz piszemy petle while bo model rerun ma korzystac z narzędzi a nie wiemy ile razy bedzie chcial skorzystac,
    # petla ma sie zakonczyć gdy done zamieni się na true a zamini sie gdy zmienna finish_reason zwróci stop 

    done = False 

    while not done:

        # teraz wszystko wywołuje sie w petli tworzymy nowe zapytanie modelu do api co wywołanie narzędzia, bo model gdy zamowi narzedzie to musi odesłać wynik do api openAI
        # dopoki zmienna done nie zamieni sia na true, czyli kiedy finish_reason zwróci stop 
        # do modelu przekazujemy wartosc tools która jest metodą api i tam wrzucamy naszą liste tools
        
        response = openai.chat.completions.create(

            model ='gpt-4.1-mini',
            messages=messages,
            tools=tools

        )


        # do zmiennej finish reason zwracamy metode finish_reason która moze zawierac albo tool_calls, albo stop w zalznoci czy model uzyl narzedzia / nardziedzi
        # albo gdy nie chce ich uzywac i ma juz odpowiedz 


        finish_reason = response.choices[0].finish_reason


        # teraz robimy petle if else która bedzie miala inny kod w zaleznosci od tego jaki jest finish reason dla danego zapytania 

        # jesli finish_reason jest tool_calls oznacza to ze model chce 1.uzyc narzedzia np save_email albo 2 
        # wowczas zrobi 5 rzecy 


        if finish_reason == 'tool_calls':

            #1 wyciągamy całą liste message, liste odpowiedzi z odpowiedzi modelu jako obiekt zeby wyciagnac z niej liste narzędzi uztych wraz z argumentami

            message_obj = response.choices[0].message

            #2 wyciągamy z obiektu odpowiedzi modelu uzyte liste uzytych narzedzi z argumentami metodlą tool_calls

            tool_calls = message_obj.tool_calls

            #3  wywolujemy funkcje któa uruchomi odpowiednie narzedzie, wraz z argumentami z tool_calls i wynik funkcji przekazujemy do zmiennej

            result = handle_tool_calls(tool_calls) 

            # 4 do listy messages, która trafia do modelu przekazujemy cały obiekt message_obj, któy zawiera ifnoramcje jakie tools z jakimi argumentami model zamówił tu mmessages dostaje info chce uzyc tools 
            # save_email

            messages.append(message_obj)

            #5 do listy messages która trafiad o modelu przekazujemy zwrot z danej funkcji danego tools np jesli uzył save_email to return to {recorded " ok"} a ten result mamy z handle_tool_calls

            messages.extend(result)

# jesli finish reason jest inny niz  tool_calls konczymy petle
        else:

            done = True

    return response.choices[0].message.content







In [58]:
# teraz funkcja chat ona przyjmuje 2 wartosi message i history, czyli wiadmosc usera i obiekt history

# równiez odpala model w pętli tak długo az finish reason nie jest stop nastepnie zwraca odpowiedz do evalautora a ten do evalauta i w funkcju if else decyduje czy uzyc modelu rerun


def chat(message,history):

# dajemy primpt do modelu system prompt + history + message
    messages =  [{'role' : 'system', 'content' : system_prompt}] + history + [{'role' : 'user'  ,'content'  : message }]


    # teraz robimy petle po to zeby co wywolanie zapytania model mogl wyslac do api info o narzedziach

    done = False

    while not done:


        # tworzymy model z parametrem tools z listą bnaszych  toools zeby model mogl ich uzyc co petle

        response = openai.chat.completions.create(

            model='gpt-4.1-mini',
            messages=messages,
            tools=tools    
        )

        # do zmiennej finish reason przekazujemy parametr finish_reason czyli powó zakonczenia funkcji który moze byc tool_calls czyli uzylem narzedzi lbo stop czyli nie potrzebuje bnrzedzi


        finish_reason = response.choices[0].finish_reason

        # jesli finish reason jest tool_calls czyli w danej petli model uzyl narzedzi to 

        if finish_reason == 'tool_calls':

            # w 1 kolejnosci pobieramy obiekt message z response, z odpowiedzi modelu
            message_obj = response.choices[0].message

            # 2 kolejnosci pobieramy lsite narzedzi uzytych przez model wraz z ich argumentami

            tool_calls = message_obj.tool_calls

            # w 3 uruchamiamy funkcje któa odpala dane narzedzie i zwraca z niego results

            result = handle_tool_calls(tool_calls)

            # w 4 do messages apendujmy cały message_obj który zawiera info jakiej funkcji tools chce uzyc model

            messages.append(message_obj) 

            # w 5 iteracji przekazujemy wynik z funkcji hadne dool_ caals do messages

            messages.extend(result)


        else: 
            done = True
    
    # zwracamy odpowiedz do zmiennej reply, z któej korzystamy potem

    reply = response.choices[0].message.content

    # teraz czas na sprawdzenie reply uruchamiamy funkcje evalaute i przekazujemy jej wynik do evalautora, który bedzie niezbedy do uruchomienia albo nie uruchomienia funkcji rerun

    evaluation = evaluate(reply,message,history)

    # teraz odpowiedz z evalaute mamy w Evalaution i mamy dostęp do is.acceptable i feedback
    # jesli Evalaution is_acceptable is true to nie musimy się martwić, 1 model dał dobtą odpowiedzi

    # robimy if else // else uruchamnia model rerun

    if evaluation.is_acceptable:
        print('ok')
    else:
        reply = rerun(reply,message,history,evaluation.feedback)
    return reply

In [61]:

history = []

msg1 = """Firma: Tebim Pro
Branża: agencja e-commerce
Opis: tworzą sklepy internetowe na PrestaShop dla polskich firm, specjalizują się w migracji sklepów i custom development, klienci to głównie średnie sklepy e-commerce w Polsce"""

result1 =  chat(msg1,history)


print(result1)

ok
Cześć,

Zauważyliśmy, że Tebim Pro specjalizuje się w tworzeniu sklepów na PrestaShop oraz migracji i custom developmentcie dla średnich e-commerce w Polsce. W Last Agency skupiamy się na SEO/SEM/GEO i AI Search, co może idealnie uzupełniać Wasze usługi - oferujemy zwiększenie widoczności i sprzedaży sklepów online.

Chętnie nawiążemy współpracę w modelu white-label lub referral, gdzie Wy skupicie się na technicznej stronie e-commerce, a my wesprzemy Was marketingowo.

Jeśli jesteście zainteresowani, możemy umówić się na krótką rozmowę, by omówić szczegóły.

Pozdrawiam,
[Imię i nazwisko]
Last Agency
